In [ ]:
#extrahieren der canonical cluster der cdrs von unseren antikörpern aus der sabdab
#wir brauchen die sabdab_summary file 

# Liste der interessanten PDBs in lowercase
your_pdbs = set(ab_ag_uniquesequences["pdb"].str.lower())

# Lade nur bestimmte Spalten der TSV
usecols = [
    "pdb",
    "cdrh1_cluster",
    "cdrh2_cluster",
    "cdrh3_length",
    "cdrl1_cluster",
    "cdrl2_cluster",
    "cdrl3_cluster"
]

#datei laden
sabdab = pd.read_csv("sabdab_summary_all.tsv", sep='\t', usecols=usecols)
#das ist nicht die korrekte datei, da hier nicht die cluster der cdrs drin sind 
#hab bisher noch nicht die richtige gefunden auf sabdab
#eventuell pylgClassify database CSV

#Filtere direkt
sabdab_filtered = sabdab[sabdab["pdb"].str.lower().isin(your_pdbs)].copy()

#Ergebnis
print(f"Anzahl PDBs mit Canonical Clusters in deinem Datensatz: {len(sabdab_filtered)}")
print(sabdab_filtered.head())

In [ ]:
#ARI, NMI oder V-measure als Quantitative Maße für die Qualität deines ESM-Clusterings im Vergleich zu Canonical Clustern
#Werte liegen zwsichen 0 (schlechte überinstimmung) und 1 (perfekte Überinstimmung)

In [ ]:
# CDR_list muss ersetzt werden mit ab_ag_annotated aufgereinigt?
# datei aus esmc
# datei mit referenz cluster finden - ascheinend auf sabdab für einzelene pdb einträge möglich aber find ich nicht?
# alternativ: PyIgClassify2 (offizielle quelle für cdr): https://dunbrack.fccc.edu/pyigclassify/

import pandas as pd
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, v_measure_score
from sklearn.preprocessing import LabelEncoder

def compare_esm_vs_canonical(canonical_df, esm_df, cdr_list=["CDR-H1", "CDR-H2", "CDR-H3", "CDR-L1", "CDR-L2", "CDR-L3"]):
    """
    Vergleicht Canonical Cluster vs. ESM-Cluster für jede CDR-Region separat.
    
    Parameter:
        canonical_df: DataFrame mit Spalten ['PDB_ID', 'Chain', 'CDR', 'Canonical_Cluster']
        esm_df: DataFrame mit Spalten ['PDB_ID', 'Chain', 'CDR', 'ESM_Cluster_Label']
        cdr_list: Liste der CDRs, die verglichen werden sollen

    Rückgabe:
        DataFrame mit ARI, NMI und V-Measure pro CDR
    """
    results = []

    # Iteriere über alle gewünschten CDRs
    for cdr in cdr_list:
        # Filtere beide DataFrames auf die aktuelle CDR
        canon_cdr = canonical_df[canonical_df["CDR"] == cdr].copy()
        esm_cdr = esm_df[esm_df["CDR"] == cdr].copy()

        # Erstelle Matching-Key
        canon_cdr["key"] = canon_cdr["PDB_ID"].str.lower() + "_" + canon_cdr["Chain"]
        esm_cdr["key"] = esm_cdr["PDB_ID"].str.lower() + "_" + esm_cdr["Chain"]

        # Gemeinsame Keys
        common_keys = set(canon_cdr["key"]) & set(esm_cdr["key"])
        if len(common_keys) < 5:
            continue  # Zu wenige gemeinsame Daten → überspringen

        canon_common = canon_cdr[canon_cdr["key"].isin(common_keys)].sort_values("key")
        esm_common = esm_cdr[esm_cdr["key"].isin(common_keys)].sort_values("key")

        # Labels extrahieren
        le = LabelEncoder()
        canon_labels = le.fit_transform(canon_common["Canonical_Cluster"])
        esm_labels = esm_common["ESM_Cluster_Label"].tolist()

        # Metriken berechnen
        ari = adjusted_rand_score(canon_labels, esm_labels)
        nmi = normalized_mutual_info_score(canon_labels, esm_labels)
        vm = v_measure_score(canon_labels, esm_labels)

        results.append({
            "CDR": cdr,
            "Anzahl_PDBs": len(common_keys),
            "ARI": round(ari, 4),
            "NMI": round(nmi, 4),
            "V-Measure": round(vm, 4)
        })

    # Rückgabe als DataFrame
    return pd.DataFrame(results)


In [ ]:
#funktion ausführen
vergleich_df = compare_esm_vs_canonical(canonical_df, esm_clusters_df)
print(vergleich_df)

In [ ]:
#die datei mit referenz clsutern enthälte alle möglichen pdb eintrräge
#filtern nach denen die in unserem datensatz sind

#Normalisiere PDB IDs auf Kleinbuchstaben (wichtig für Matching!)
ab_ag_uniquesequences["pdb_lower"] = ab_ag_uniquesequences["pdb"].str.lower()
pyig_df["PDB_ID_lower"] = pyig_df["PDB_ID"].str.lower()

#Erstellen eindeutiger Schlüssel zum Vergleich: PDB + Chain
ab_keys = set(ab_ag_uniquesequences["pdb_lower"] + "_" + ab_ag_uniquesequences["chain"])
pyig_df["match_key"] = pyig_df["PDB_ID_lower"] + "_" + pyig_df["Chain"]

#Filter: Nur Zeilen behalten, die auch in unserem Datensatz vorkommen
pyig_filtered = pyig_df[pyig_df["match_key"].isin(ab_keys)].copy()

print(f"Originale Anzahl PyIgClassify-Einträge: {len(pyig_df)}")
print(f"Übereinstimmende Einträge mit deinem Datensatz: {len(pyig_filtered)}")

print(pyig_filtered.head())